# InputLine and InputAssumption

This notebook demonstrates `InputLine` and `InputAssumption` — the two types that turn a model's
class definition into a reusable template where scenario values are supplied at instantiation.

| Type | What it replaces | Value supplied |
|------|-----------------|----------------|
| `InputLine` | `FixedLine` | dict mapping period → value, passed as a kwarg |
| `InputAssumption` | `Assumption` | scalar, passed as a kwarg |

Both are declared in the class body; both are provided when you call `ModelClass(periods=[...], ...)`.


In [ ]:
from pyproforma import (
    Assumption,
    FixedLine,
    FormulaLine,
    InputAssumption,
    InputLine,
    ProformaModel,
    Format,
)

## Define the Model

A simple coffee-shop P&L model. The **fixed costs** are the same in every scenario
(baked into the class with `FixedLine` and `Assumption`). The **revenue** and **variable
cost ratio** change between scenarios, so they are declared with `InputLine` and
`InputAssumption` — the caller must supply them at instantiation.


In [ ]:
class CoffeeShopModel(ProformaModel):
    # Fixed — same in every scenario
    rent = FixedLine(
        values={2024: 60_000, 2025: 62_000, 2026: 64_000},
        label="Rent",
        value_format=Format.CURRENCY_NO_DECIMALS,
    )
    staff_costs = Assumption(value=120_000, label="Annual Staff Costs")

    # Scenario inputs — supplied at instantiation
    revenue = InputLine(
        label="Revenue",
        value_format=Format.CURRENCY_NO_DECIMALS,
    )
    cogs_ratio = InputAssumption(label="Cost of Goods Ratio")

    # Calculated lines
    cogs = FormulaLine(
        formula=lambda li, t: li.revenue[t] * li.cogs_ratio,
        label="Cost of Goods Sold",
        value_format=Format.CURRENCY_NO_DECIMALS,
    )
    gross_profit = FormulaLine(
        formula=lambda li, t: li.revenue[t] - li.cogs[t],
        label="Gross Profit",
        value_format=Format.CURRENCY_NO_DECIMALS,
    )
    total_fixed_costs = FormulaLine(
        formula=lambda li, t: li.rent[t] + li.staff_costs,
        label="Total Fixed Costs",
        value_format=Format.CURRENCY_NO_DECIMALS,
    )
    ebitda = FormulaLine(
        formula=lambda li, t: li.gross_profit[t] - li.total_fixed_costs[t],
        label="EBITDA",
        value_format=Format.CURRENCY_NO_DECIMALS,
    )

## Instantiate — Base Scenario

`revenue` (an `InputLine`) takes a `{period: value}` dict.  
`cogs_ratio` (an `InputAssumption`) takes a scalar.


In [ ]:
base = CoffeeShopModel(
    periods=[2024, 2025, 2026],
    revenue={2024: 400_000, 2025: 440_000, 2026: 480_000},
    cogs_ratio=0.35,
)

base.tables.line_items(include_label=True, include_total_row=False).show()

## Instantiate — Optimistic Scenario

Same class, different inputs — higher revenue, tighter cost of goods.


In [ ]:
optimistic = CoffeeShopModel(
    periods=[2024, 2025, 2026],
    revenue={2024: 520_000, 2025: 580_000, 2026: 650_000},
    cogs_ratio=0.30,
)

optimistic.tables.line_items(include_label=True, include_total_row=False).show()

## Required vs Optional Inputs

`InputAssumption` can have a **default** value — the caller may override it but doesn't have to.
Without a default, the assumption is required and omitting it raises a clear error.


In [ ]:
# Omitting a required InputLine raises an informative error
try:
    CoffeeShopModel(periods=[2024], cogs_ratio=0.35)  # missing revenue
except TypeError as e:
    print(e)

In [ ]:
# Omitting a required InputAssumption also raises a clear error
try:
    CoffeeShopModel(
        periods=[2024],
        revenue={2024: 400_000},  # missing cogs_ratio
    )
except TypeError as e:
    print(e)

In [ ]:
class ModelWithDefault(ProformaModel):
    revenue = FixedLine(values={2024: 500_000}, label="Revenue", value_format=Format.CURRENCY_NO_DECIMALS)
    cogs_ratio = InputAssumption(default=0.35, label="COGS Ratio")
    cogs = FormulaLine(
        formula=lambda li, t: li.revenue[t] * li.cogs_ratio,
        label="COGS",
        value_format=Format.CURRENCY_NO_DECIMALS,
    )

# Uses the default
m_default = ModelWithDefault(periods=[2024])
print(f"Using default cogs_ratio: {m_default.av.cogs_ratio}")

# Override the default
m_override = ModelWithDefault(periods=[2024], cogs_ratio=0.28)
print(f"Overridden cogs_ratio:    {m_override.av.cogs_ratio}")

## Accessing Values

Once instantiated, `InputLine` and `InputAssumption` values are accessed the same way as
`FixedLine` and `Assumption` — through `model.li` and `model.av`.


In [ ]:
# InputLine values via model.li
print(f"Base revenue 2025:       {base.li.revenue[2025]:,.0f}")

# InputAssumption values via model.av
print(f"Base cogs_ratio:         {base.av.cogs_ratio}")
print(f"Optimistic cogs_ratio:   {optimistic.av.cogs_ratio}")

# Calculated values work the same as always
print(f"Base EBITDA 2026:        {base.li.ebitda[2026]:,.0f}")
print(f"Optimistic EBITDA 2026:  {optimistic.li.ebitda[2026]:,.0f}")